# Detail-Auswertung

## Imports and constants

In [1]:
# Imports / global contants

# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib import figure
import math
import numpy as np
import seaborn as sns
from scipy.misc import electrocardiogram
from scipy.signal import find_peaks
import gc
import pylab

from pandas.plotting import parallel_coordinates

%matplotlib inline

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"
export_interactions = "../export/interaction/"

path = "../data/"

In [2]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy.csv', sep=";")

# retrieve labels for results
states = df_cleaned[
    ((df_cleaned["posX"] == " COMPLETED") | 
    (df_cleaned["posX"] == " FAILED") | 
    (df_cleaned["posX"] == " TERMINATED") | 
    (df_cleaned["posX"] == " START"))
  ]

# insert labels
df_cleaned["Result"] = states["posX"]

# fill (bottom-up)
df_cleaned["Result"].fillna(method='bfill', inplace=True)

# drop na (end)
df_cleaned = df_cleaned.dropna(subset=["Result"])

# drop start (between trials)
df_cleaned = df_cleaned.drop(df_cleaned[(df_cleaned["Result"] == " START") & (df_cleaned["posX"] != " START")].index)

df_cleaned["Block"] = (df_cleaned["TaskNo"] / 18).apply(np.ceil).astype(int)
df_cleaned["TaskNo"] = (df_cleaned["TaskNo"] - ((df_cleaned["Block"] - 1) * 18)).astype(int)
df_cleaned["TrialIdx"] = df_cleaned["TrialIdx"].astype(int)   
df_cleaned["TargetLayers"] = df_cleaned["TargetLayers"].str.split(",")
df_cleaned["Target"] = df_cleaned.apply(lambda item: item["TargetLayers"][item["TrialIdx"]], axis = 1).astype(int)
df_cleaned["Target_Relative"] = df_cleaned["Target"].astype(float) / df_cleaned["LayerCount"].astype(float)

df_cleaned.reset_index(inplace=True)

df_cleaned.rename(columns={ df_cleaned.columns[0]:"SampleIdx_global", df_cleaned.columns[1]:"SampleIdx_Proband", "TrialIdx":"Trial", "LayerCount":"NumLayers", "TaskNo":"Task" }, inplace=True)

display(df_cleaned)

df_cleaned.to_csv(rf'{export}cleanedStudy_interactionOnly.csv', sep= ";")



,SampleIdx_global,SampleIdx_Proband,DateTime,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,posX,...,posZ,TimeStamp,iteractionType,currentLayer,Proband,shifted,Result,Block,Target,Target_Relative
0,135,135,2021-05-10T12:13:56.584Z,VIEW,direct,1,"[4, 5, 3, 1, 2]",6,0,START,...,NaN,NaN,NaN,NaN,1,1.0,START,1,4,0.666667
1,136,136,2021-05-10T12:13:56.602Z,INTERACTION,direct,1,"[4, 5, 3, 1, 2]",6,0,-,...,-,-,-,2,1,1.0,COMPLETED,1,4,0.666667
2,137,137,2021-05-10T12:13:56.633Z,INTERACTION,direct,1,"[4, 5, 3, 1, 2]",6,0,-,...,-,-,-,2,1,1.0,COMPLETED,1,4,0.666667
3,138,138,2021-05-10T12:13:56.676Z,INTERACTION,direct,1,"[4, 5, 3, 1, 2]",6,0,-,...,-,-,-,-,1,1.0,COMPLETED,1,4,0.666667
4,139,139,2021-05-10T12:13:56.697Z,INTERACTION,direct,1,"[4, 5, 3, 1, 2]",6,0,-,...,-,-,-,-,1,1.0,COMPLETED,1,4,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1535230,2360983,90204,2021-06-30T12:37:15.245Z,INTERACTION,widening,18,"[4, 5, 2, 3, 1]",6,4,0.252979726,...,-0.9999999,637606606352421800,1,6,24,54.0,FAILED,3,1,0.166667
1535231,2360984,90205,2021-06-30T12:37:15.291Z,INTERACTION,widening,18,"[4, 5, 2, 3, 1]",6,4,0.231738284,...,-0.9999999,637606606352861800,1,6,24,54.0,FAILED,3,1,0.166667
1535232,2360985,90206,2021-06-30T12:37:15.323Z,INTERACTION,widening,18,"[4, 5, 2, 3, 1]",6,4,0.231738284,...,-0.9999999,637606606353161900,1,6,24,54.0,FAILED,3,1,0.166667
1535233,2360986,90207,2021-06-30T12:37:15.361Z,INTERACTION,widening,18,"[4, 5, 2, 3, 1]",6,4,0.240411818,...,-0.9999999,637606606353551700,1,6,24,54.0,FAILED,3,1,0.166667


---

## helper function to get layer borders

---

In [3]:
def getLayerBorders(layers, mapping):
    layerdata = pd.read_csv(rf'{path}meta/depthlayers.csv', sep=";")

    layer = layerdata[(layerdata.iloc[:,0] == layers) & (layerdata.iloc[:, 1] == mapping)]

    return np.array(layer.iloc[:,3].str.split("|").iloc[0], float)

In [4]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly.csv', sep=";")

# proband = 24
# task = 18
# block = 3
# trial = 3

def drawSingleInteractionGraph(proband, block, task, trial, show=False):

    exp_data = df_cleaned[(df_cleaned["Proband"] == proband) & (df_cleaned["Task"] == task) & (df_cleaned["Block"] == block) & (df_cleaned["Trial"] == trial)].reset_index(drop=True)

    z_data = exp_data["posZ"].replace("-", 0).astype(float)

    target = exp_data["Target"][0]
    num_layers = exp_data["NumLayers"][0]
    mapping = exp_data["mappingMethod"][0]

    borders = getLayerBorders(num_layers, mapping)

    miny = [-borders[target+1]]*len(z_data)
    maxy = [-borders[target]]*len(z_data)

    hold = exp_data[exp_data["posX"] == " HOLD"].index

    ex_max, _ = find_peaks(z_data, prominence=0.05)
    ex_min, _ = find_peaks(-z_data, prominence=0.05)

    peaks = np.concatenate([ex_max,ex_min])

    fig,ax = plt.subplots(1, 1, tight_layout=True, figsize=(25,8), clear=True)

    ax.plot(exp_data.index, z_data, label='Depth')
    ax.plot(peaks, z_data[peaks], "o")

    plt.vlines(x= hold, ymin=-1, ymax=0, linestyles="--")

    ax.plot(miny, linestyle=":")
    ax.plot(maxy, linestyle=":")
    
    plt.suptitle(f'Proband {proband}: Block {block} | Task {task} | Trial {trial}',fontsize=20, y=1)
    plt.title(f'{num_layers} Layers ({mapping}) | Target: {target}',fontsize=16)

    plt.savefig(rf'{export_interactions}single/Interaction_p{proband}_{block}_{task}_{trial}.jpg')
    plt.savefig(rf'{export_interactions}single/Interaction_p{proband}_{block}_{task}_{trial}.svg')

    if show:
        plt.show()
    else:   
        plt.close(fig)          

## extract all graphs for specific layer / target / mapping config

In [5]:
def drawInteractionGraph(num_layers, target, mapping, show=False):

   exp_data_complete = df_cleaned[(df_cleaned["mappingMethod"] == mapping) & (df_cleaned["Target"] == target) & (df_cleaned["NumLayers"] == num_layers)].reset_index(drop=True).groupby(["Proband", "Task", "Block", "Trial"])

   borders = getLayerBorders(num_layers, mapping)

   fig,ax = plt.subplots(1, 1, tight_layout=True, figsize=(25,8), clear=True)

   z_data = pd.DataFrame()
   peaks = pd.DataFrame()
   hold = []

   for name, group in exp_data_complete:

      norm = group.reset_index(drop=True)

      newData = norm["posZ"].replace("-", 0).astype(float)
      z_data = pd.concat([z_data, newData], axis=1, ignore_index=True)

      ax.plot(newData.index, newData, label=name)

      ex_max, _ = find_peaks(newData, prominence=0.05)
      ex_min, _ = find_peaks(-newData, prominence=0.05)

      newPeaks = np.concatenate([ex_max,ex_min])
      peaks = pd.concat([peaks, pd.DataFrame(newPeaks)], axis=1, ignore_index=True)
      
      ax.plot(newPeaks, newData[newPeaks], "o")

      newHold = np.array(norm[norm["posX"] == " HOLD"].index)

      hold = np.concatenate([hold, newHold])

   plt.vlines(x= hold, ymin=-1, ymax=0, linestyles="--")

   miny = [-borders[target+1]]*len(z_data)
   maxy = [-borders[target]]*len(z_data)

   ax.plot(miny, linestyle=":")
   ax.plot(maxy, linestyle=":")

   plt.suptitle(f'{num_layers} Layers ({mapping}) | Target: {target}',fontsize=20, y=1)

   plt.savefig(rf'{export_interactions}allInteractions_{target}-{num_layers}_{mapping}.jpg')
   plt.savefig(rf'{export_interactions}allInteractions_{target}-{num_layers}_{mapping}.svg')
     
   if show:
      plt.show()
   else:
      plt.close(fig)

---
## draw ALL interaction graphs per configuration (Layers/Target/MappingMethod)
---

In [ ]:
# num layers
all_layers = [6,9,12,15,18,21]

# mapping
mappings = [" direct", " densening", " widening"]

plt.ioff()
plt.close('all')

for num in all_layers:
    all_targets = np.array(df_cleaned[df_cleaned["NumLayers"] == num]["Target"].drop_duplicates().sort_values())
    gc.collect()
    print(f'plotting config for {num} layers')
    for target in all_targets:
        for map in mappings:
            drawInteractionGraph(num, target, map)

plt.ion()

---
## Draw all Single Interaction Graphs

---

In [ ]:
all_probands = np.array(df_cleaned["Proband"].drop_duplicates().sort_values())
all_blocks = np.array(df_cleaned["Block"].drop_duplicates().sort_values())
all_tasks = np.array(df_cleaned["Task"].drop_duplicates().sort_values())
all_trials = np.array(df_cleaned["Trial"].drop_duplicates().sort_values())

print(f'Starting to plot all single graphs for {len(all_probands)} Probands')
plt.ioff()
plt.close('all')

for p in all_probands:
    gc.collect()    
    for b in all_blocks:       
        for t in all_tasks:
            for tr in all_trials:
                drawSingleInteractionGraph(p,b,t,tr)
    print(f'Finished Proband {p}')  

plt.ion()

--- 
## Statistics for peaks

| NumLayers | MappingMethod | TargetLayer | Peaks_Z           | Peaks_total | Peaks_lower  | Peaks_inside | Peaks_deeper| Peaks_before_hold | Peaks_after_hold | Result    |
| --------- | ------------- | ----------- | ----------------- | ----------- | ------------ | ------------ | ----------- | ------------------| ---------------- | --------- | 
| 21        | direct        | 6           | [-0.1, -0.5, ...] | 6           | 3            | 1            | 2           | 6                 | 0                | COMPLETED |

---

In [125]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly.csv', sep=";")

def extractAllPeaks(num_layers, mapping, target):
   exp_data_complete = df_cleaned[(df_cleaned["mappingMethod"] == mapping) & (df_cleaned["Target"] == target) & (df_cleaned["NumLayers"] == num_layers)].reset_index(drop=True).groupby(["Proband", "Task", "Block", "Trial"])

   borders = getLayerBorders(num_layers, mapping)

   complete = pd.DataFrame()

   for name, group in exp_data_complete:
      
      result = pd.DataFrame()      

      norm = group.reset_index(drop=True)

      z_data = np.array(norm["posZ"].replace("-", 0).astype(float), float)  

      ex_max, _ = find_peaks(z_data, prominence=0.05)
      ex_min, _ = find_peaks(-z_data, prominence=0.05)
      
      peaks = []
      # if len(ex_min) + len(ex_max) > 0:
      peaks = np.concatenate([ex_max,ex_min])

      pos = norm[norm["posX"] == " HOLD"]
      
      if len(pos) == 1:
         hold = np.array(pos.index)

      else: #hold.isnull():
         hold = np.array(norm.index[-1])

      peaks_z = pd.DataFrame(pd.DataFrame(peaks).apply(lambda item: norm.iloc[item.index]["posZ"])[0].replace(to_replace='-', value='0').astype(float).fillna(0))
           
      result["Peaks_Z"] = [peaks_z[0].values]
      result["Peaks_Total"] = len(peaks)

      result["Result"] = norm["Result"][1]

      result["HOLD_idx"] = [hold]

      before = peaks[peaks >= hold]
      after = peaks[peaks < hold]

      result["Peaks_AfterHold"] = len(before)
      result["Peaks_AfterHold_Values"] = [before]
      result["Peaks_BeforeHold"] = len(after)
      result["Peaks_BeforeHold_Values"] = [after]

      lower = peaks_z[peaks_z[0] > -borders[target+1]]
      deeper = peaks_z[peaks_z[0] < -borders[target]]

      result["Peaks_lower"] = len(lower)
      result["Peaks_lower_Values"] = [lower[0].values]
      result["Peaks_deeper"] = len(deeper)
      result["Peaks_deeper_Values"] = [deeper[0].values]

      result["Peaks_between"] = result["Peaks_Total"] - (result["Peaks_lower"] + result["Peaks_deeper"])

      result["Peaks_Z"] = result["Peaks_Z"]
      
      complete = pd.concat([complete, result])
   

   complete["NumLayers"] = num_layers
   complete["MappingMethod"] = mapping
   complete["TargetLayer"] = target

   return complete
   
# test
peaks_df = extractAllPeaks(6, " direct", 1)
display(peaks_df)

,Peaks_Z,Peaks_Total,Result,HOLD_idx,Peaks_AfterHold,Peaks_AfterHold_Values,Peaks_BeforeHold,Peaks_BeforeHold_Values,Peaks_lower,Peaks_lower_Values,Peaks_deeper,Peaks_deeper_Values,Peaks_between,NumLayers,MappingMethod,TargetLayer
0,"[0.0, -0.5171052]",2,TERMINATED,[96],0,[],2,"[28, 71]",1,[0.0],1,[-0.5171052],0,6,direct,1
0,"[0.0, -0.118420973]",2,COMPLETED,[105],0,[],2,"[18, 57]",2,"[0.0, -0.118420973]",1,[-0.118420973],-1,6,direct,1
0,"[0.0, -0.363157839]",2,COMPLETED,[106],0,[],2,"[17, 49]",1,[0.0],1,[-0.363157839],0,6,direct,1
0,[0.0],1,COMPLETED,[78],0,[],1,[22],1,[0.0],0,[],0,6,direct,1
0,"[0.0, -0.264473617, -0.288157851, -0.323684067]",4,COMPLETED,[168],0,[],4,"[37, 116, 24, 61]",1,[0.0],3,"[-0.264473617, -0.288157851, -0.323684067]",0,6,direct,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,"[0.0, -0.525, -0.5802631]",3,COMPLETED,[120],0,[],3,"[29, 12, 55]",1,[0.0],2,"[-0.525, -0.5802631]",0,6,direct,1
0,"[0.0, -0.7342105, -0.76578933, -0.7736841]",4,COMPLETED,[144],0,[],4,"[21, 85, 50, 95]",1,[0.0],3,"[-0.7342105, -0.76578933, -0.7736841]",0,6,direct,1
0,"[0.0, -0.232894585]",2,COMPLETED,[101],0,[],2,"[16, 46]",2,"[0.0, -0.232894585]",1,[-0.232894585],-1,6,direct,1
0,"[0.0, -0.343421]",2,COMPLETED,[107],0,[],2,"[18, 50]",1,[0.0],1,[-0.343421],0,6,direct,1


In [135]:
all_layers = np.array(df_cleaned["NumLayers"].drop_duplicates().sort_values())
all_mappings = np.array(df_cleaned["mappingMethod"].drop_duplicates().sort_values())

result_all = pd.DataFrame()

for l in all_layers:  
    for m in all_mappings:
        all_targets = np.array(df_cleaned[(df_cleaned["NumLayers"] == l) & (df_cleaned["mappingMethod"] == m)]["Target"].drop_duplicates().sort_values())
        print(f'{l} layers [{m}] - targets: {all_targets}')
        for t in all_targets:
            # print(f'{l}, {m}, {t}')
            p = extractAllPeaks(l, m, t)            
            result_all = pd.concat([result_all, p])



result_all["Peaks_Z"] = result_all["Peaks_Z"].apply(lambda item: np.array2string(item))
result_all["Peaks_lower_Values"] = result_all["Peaks_lower_Values"].apply(lambda item: np.array2string(item))
result_all["Peaks_deeper_Values"] = result_all["Peaks_deeper_Values"].apply(lambda item: np.array2string(item))

display(result_all)

result_all.to_csv(rf'{export}all_peaks.csv', sep= ";")

test = pd.read_csv(rf'{export}all_peaks.csv', sep= ";")

test["Peaks_Z"] = test["Peaks_Z"].apply(lambda item: np.array(item))
test["Peaks_lower_Values"] = test["Peaks_lower_Values"].apply(lambda item: np.array(item))
test["Peaks_deeper_Values"] = test["Peaks_deeper_Values"].apply(lambda item: np.array(item))

display(test)


6 layers [ densening] - targets: [1 2 3 4 5]
6 layers [ direct] - targets: [1 2 3 4 5]
6 layers [ widening] - targets: [1 2 3 4 5]
9 layers [ densening] - targets: [1 2 5 7 8]
9 layers [ direct] - targets: [1 2 5 7 8]
9 layers [ widening] - targets: [1 2 5 7 8]
12 layers [ densening] - targets: [ 1  3  6  9 11]
12 layers [ direct] - targets: [ 1  3  6  9 11]
12 layers [ widening] - targets: [ 1  3  6  9 11]
15 layers [ densening] - targets: [ 1  4  8 11 14]
15 layers [ direct] - targets: [ 1  4  8 11 14]
15 layers [ widening] - targets: [ 1  4  8 11 14]
18 layers [ densening] - targets: [ 1  5  9 14 17]
18 layers [ direct] - targets: [ 1  5  9 14 17]
18 layers [ widening] - targets: [ 1  5  9 14 17]
21 layers [ densening] - targets: [ 1  5 11 16 20]
21 layers [ direct] - targets: [ 1  5 11 16 20]
21 layers [ widening] - targets: [ 1  5 11 16 20]


,Peaks_Z,Peaks_Total,Result,HOLD_idx,Peaks_AfterHold,Peaks_AfterHold_Values,Peaks_BeforeHold,Peaks_BeforeHold_Values,Peaks_lower,Peaks_lower_Values,Peaks_deeper,Peaks_deeper_Values,Peaks_between,NumLayers,MappingMethod,TargetLayer
0,[ 0. -0.22894727],2,TERMINATED,[114],0,[],2,"[14, 51]",2,[ 0. -0.22894727],1,[-0.22894727],-1,6,densening,1
0,[ 0. -0.50921047],2,COMPLETED,[101],0,[],2,"[12, 47]",1,[0.],1,[-0.50921047],0,6,densening,1
0,[ 0. -0.55263156 -0.37894732],3,TERMINATED,[70],0,[],3,"[13, 46, 50]",2,[ 0. -0.37894732],2,[-0.55263156 -0.37894732],-1,6,densening,1
0,[0.],1,TERMINATED,[70],0,[],1,[18],1,[0.],0,[],0,6,densening,1
0,[ 0. -0.5644736],2,COMPLETED,[96],0,[],2,"[15, 46]",1,[0.],1,[-0.5644736],0,6,densening,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,[],0,COMPLETED,[162],0,[],0,[],0,[],0,[],0,21,widening,20
0,[ 0. -0.6947367 -0.7578946],3,COMPLETED,[208],0,[],3,"[26, 3, 158]",3,[ 0. -0.6947367 -0.7578946],0,[],0,21,widening,20
0,[ 0. -0.31973675 -0.31578943],3,COMPLETED,[225],0,[],3,"[15, 174, 138]",3,[ 0. -0.31973675 -0.31578943],0,[],0,21,widening,20
0,[0.],1,TERMINATED,[148],0,[],1,[15],1,[0.],0,[],0,21,widening,20


,Unnamed: 0,Peaks_Z,Peaks_Total,Result,HOLD_idx,Peaks_AfterHold,Peaks_AfterHold_Values,Peaks_BeforeHold,Peaks_BeforeHold_Values,Peaks_lower,Peaks_lower_Values,Peaks_deeper,Peaks_deeper_Values,Peaks_between,NumLayers,MappingMethod,TargetLayer
0,0,[ 0. -0.22894727],2,TERMINATED,[114],0,[],2,[14 51],2,[ 0. -0.22894727],1,[-0.22894727],-1,6,densening,1
1,0,[ 0. -0.50921047],2,COMPLETED,[101],0,[],2,[12 47],1,[0.],1,[-0.50921047],0,6,densening,1
2,0,[ 0. -0.55263156 -0.37894732],3,TERMINATED,[70],0,[],3,[13 46 50],2,[ 0. -0.37894732],2,[-0.55263156 -0.37894732],-1,6,densening,1
3,0,[0.],1,TERMINATED,[70],0,[],1,[18],1,[0.],0,[],0,6,densening,1
4,0,[ 0. -0.5644736],2,COMPLETED,[96],0,[],2,[15 46],1,[0.],1,[-0.5644736],0,6,densening,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,0,[],0,COMPLETED,[162],0,[],0,[],0,[],0,[],0,21,widening,20
6206,0,[ 0. -0.6947367 -0.7578946],3,COMPLETED,[208],0,[],3,[ 26 3 158],3,[ 0. -0.6947367 -0.7578946],0,[],0,21,widening,20
6207,0,[ 0. -0.31973675 -0.31578943],3,COMPLETED,[225],0,[],3,[ 15 174 138],3,[ 0. -0.31973675 -0.31578943],0,[],0,21,widening,20
6208,0,[0.],1,TERMINATED,[148],0,[],1,[15],1,[0.],0,[],0,21,widening,20
